# CS781 Phase 5 — Experiment 3: Model2 + AdaLoRA (Fast + Refit + Ensemble)

**Task:** Legal Statute Identification — 100-label multilabel classification  
**Phase-1 foundation:** Model2 — 13-layer decoder-only Transformer (~79.7M), d=640, 10 heads, FFN=2560, sinusoidal positional encoding, pre-LN, GELU, tied output embedding  
**Adaptation:** **AdaLoRA — Adaptive Budget Allocation for Parameter-Efficient Fine-Tuning**

This experiment preserves the fast Phase-5 pipeline used in Experiment 2 so the main methodological change is **Model2 + AdaLoRA**.

### AdaLoRA implementation used here

For each selected frozen projection \(W_0\), the trainable update is parameterized in SVD-like form:

\[
W = W_0 + P\,\mathrm{diag}(E)\,Q
\]

where rank-one triplets are globally ranked by their importance. The run starts with a larger total rank budget, warms up, **shrinks the budget with a cubic schedule**, and then freezes the final rank allocation for the end of training.

Defaults:

- initial average rank = **12**
- target average rank = **8**
- warm-up = first **10%** of optimizer updates
- final allocation fixed for last **20%**
- budget reallocation every **10 optimizer updates**
- selected Model2 blocks = **6–12** (top 7 blocks)
- adapted matrices = fused attention `qkv`, attention `out_proj`, FFN input and output projections

This is materially different from fixed-rank LoRA: rank budget is dynamically reallocated among matrices during training.

Reference: Zhang et al., *Adaptive Budget Allocation for Parameter-Efficient Fine-Tuning (AdaLoRA)*, ICLR 2023.

### Submission outputs

The full run creates three core Experiment-3 ZIPs:

1. **Heldout AdaLoRA**
2. **Fresh TRAIN+DEV refit**
3. **Heldout + refit probability ensemble**

An optional last cell can also blend Exp2 and Exp3 probabilities if Exp2 `.npz` files are uploaded.


### v2 balanced changes

After reviewing the modified notebook, this version keeps the useful changes but stays close to the canonical AdaLoRA setup:

- **AdaLoRA LR = 8e-4**
- canonical **initial rank 12 → target average rank 8**
- canonical **alpha = 16**
- `RUN_REFIT=True` by default, but can be switched off for a quick debug run
- refit, ensemble, manifest and optional cross-experiment blend cells are guarded so the notebook remains runnable either way

The rank/alpha settings were intentionally not increased to 16/10/24 because that raises trainable capacity and compute without a clear validation advantage.


## Kaggle inputs

Attach:

1. your Model2 Phase-1 **`dec_full_best*.pt`**
2. the exact Model2 **`tokenizer.json`**

The notebook auto-searches `/kaggle/input`. If it chooses the wrong file, set the override paths in Cell 1.


In [4]:
# CELL 1 — imports, exact Model2 constants, AdaLoRA + fast Phase-5 config
import os, re, gc, json, math, time, random, hashlib, zipfile, shutil, heapq, contextlib, subprocess, sys, importlib.util
from pathlib import Path
from collections import Counter, defaultdict

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import f1_score, accuracy_score
from huggingface_hub import hf_hub_download, login

SEED = 2345
NOTEBOOK_VERSION = "exp3_model2_adalora_fast_v2_balanced_refit_ensemble"

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)
if DEVICE.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
torch.set_float32_matmul_precision("high")

ROOT = Path("/kaggle/working/phase5_exp3_model2_adalora_fast_v2_balanced")
CACHE_DIR = ROOT / "cache"
ROOT.mkdir(parents=True, exist_ok=True)
CACHE_DIR.mkdir(parents=True, exist_ok=True)

# ---- exact Phase-1 Model2 constants ----
VOCAB_SIZE = 24_576
D_MODEL = 640
N_LAYERS = 13
N_HEADS = 10
FFN_DIM = 2560
DROPOUT = 0.0
MAX_POSITIONS = 2048

# ---- Phase-5 ----
N_LABELS = 100
N_CHUNKS = 3
CHUNK_LEN = 320
CHAR_WINDOW = 5000
VIEW_NAMES = ["head", "statute_evidence", "tail"]

# ---- AdaLoRA ----
# Lower six blocks remain fully frozen and execute under no_grad.
ADAPT_START_BLOCK = 6
ADAPTED_BLOCKS = list(range(ADAPT_START_BLOCK, N_LAYERS))
ADALORA_TARGETS = ("qkv", "out_proj", "ff_in", "ff_out")

ADALORA_INIT_RANK = 12
ADALORA_TARGET_AVG_RANK = 8
ADALORA_ALPHA = 16.0
ADALORA_INIT_STD = 0.02
ADALORA_BETA1 = 0.85
ADALORA_BETA2 = 0.85
ADALORA_MASK_INTERVAL = 10
ADALORA_WARMUP_FRAC = 0.10
ADALORA_FINAL_FRAC = 0.20
ADALORA_ORTH_COEF = 0.01

# ---- training ----
MICRO_BATCH_SIZE = 4
DEV_BATCH_SIZE = 16
GRAD_ACCUM_STEPS = 4        # effective document batch = 16
EPOCHS = 2
ADALORA_LR = 8e-4  # official AdaLoRA repo uses 8e-4 in its QA example
HEAD_LR = 8e-4
WEIGHT_DECAY = 0.01
WARMUP_RATIO = 0.06
EMA_DECAY = 0.998
GRAD_CLIP = 1.0
USE_AMP = True
NUM_WORKERS = 2

FAST_CAUSAL_NO_MASK = True

# Refit remains enabled by default because we want the extra refit + ensemble submissions.
# Set False only for a quick heldout-only debug run.
RUN_REFIT = True
REFIT_ENSEMBLE_WEIGHT = 0.50
REFIT_EPOCHS = None

# ---- threshold tuning ----
THR_GRID = np.round(np.arange(0.02, 0.8001, 0.01), 3)
SHRINK_TAUS = [25.0, 50.0, 100.0, 200.0]
CV_REPEATS = 2

MODEL2_CKPT_OVERRIDE = None
TOKENIZER_OVERRIDE = None

def sha256_file(path, chunk=1<<20):
    h=hashlib.sha256()
    with open(path,"rb") as f:
        while True:
            b=f.read(chunk)
            if not b: break
            h.update(b)
    return h.hexdigest()

def norm_text(s):
    return re.sub(r"\s+"," ",str(s).lower()).strip()

def text_hash(s):
    return hashlib.sha256(norm_text(s).encode("utf-8")).hexdigest()

def atomic_torch_save(obj,path):
    path=Path(path); tmp=Path(str(path)+".tmp")
    torch.save(obj,tmp); os.replace(tmp,path)

print("ROOT:",ROOT)
print("Model2 adapted blocks:",ADAPTED_BLOCKS)
print("AdaLoRA targets:",ADALORA_TARGETS)
print("AdaLoRA rank: init",ADALORA_INIT_RANK,"-> target avg",ADALORA_TARGET_AVG_RANK)
print("Views:",VIEW_NAMES,"chunk_len:",CHUNK_LEN)


Device: cuda
GPU: Tesla T4
ROOT: /kaggle/working/phase5_exp3_model2_adalora_fast_v2_balanced
Model2 adapted blocks: [6, 7, 8, 9, 10, 11, 12]
AdaLoRA targets: ('qkv', 'out_proj', 'ff_in', 'ff_out')
AdaLoRA rank: init 12 -> target avg 8
Views: ['head', 'statute_evidence', 'tail'] chunk_len: 320


In [5]:
# CELL 2 — locate Model2 BEST checkpoint + exact tokenizer
all_files=[]
for base in ["/kaggle/input","/kaggle/working"]:
    if os.path.exists(base):
        for root,dirs,files in os.walk(base):
            for f in files:
                all_files.append(os.path.join(root,f))

ckpt_cands=[
    p for p in all_files
    if p.lower().endswith(".pt")
    and "dec_full_best" in os.path.basename(p).lower()
]
tok_cands=[
    p for p in all_files
    if os.path.basename(p).lower()=="tokenizer (1).json"
]

def asset_score(p,words):
    s=p.lower()
    return sum(10 for w in words if w in s)-len(p)/100000.0

if MODEL2_CKPT_OVERRIDE:
    MODEL2_CKPT=str(MODEL2_CKPT_OVERRIDE)
else:
    assert ckpt_cands,(
        "Could not find dec_full_best*.pt. "
        "Attach your Phase-1 Model2 BEST checkpoint."
    )
    MODEL2_CKPT=sorted(
        ckpt_cands,
        key=lambda p:asset_score(
            p,["model2","decoder-only","phase1","complete-dataset"]
        ),
        reverse=True
    )[0]

if TOKENIZER_OVERRIDE:
    TOKENIZER_FILE=str(TOKENIZER_OVERRIDE)
else:
    assert tok_cands,"Could not find tokenizer.json."
    ckpt_dir=os.path.dirname(MODEL2_CKPT).lower()
    TOKENIZER_FILE=sorted(
        tok_cands,
        key=lambda p:(
            20 if os.path.dirname(p).lower()==ckpt_dir else 0,
            asset_score(p,["model2","decoder-only","phase1"])
        ),
        reverse=True
    )[0]

print("SELECTED MODEL2:",MODEL2_CKPT)
print("SELECTED TOKENIZER:",TOKENIZER_FILE)
print("Model2 SHA256:",sha256_file(MODEL2_CKPT))
print("Tokenizer SHA256:",sha256_file(TOKENIZER_FILE))


SELECTED MODEL2: /kaggle/input/datasets/aaryant23/phase2-model2-ckpts/dec_full_best (8).pt
SELECTED TOKENIZER: /kaggle/input/datasets/aaryant23/phase2-model2-ckpts/tokenizer (1).json
Model2 SHA256: 7685624ff48e4c2fbf213f9be4f626173e52676b007b9bc50dc44b0323d3cc15
Tokenizer SHA256: 9cfbe6f0e49067f63ba71fe1013021c5de9269a2b4bc6c9ae30a72e42102106d


In [6]:
# CELL 3 — exact Phase-1 Model2 LegalBPE tokenizer
# Cell 3 — Custom byte-level BPE (same assignment design, scaled up)
BPE_SPECIALS = ["<pad>", "<unk>", "<s>", "</s>"]

class LegalBPE:
    def __init__(self, vocab_size=5000):
        self.vocab_size = int(vocab_size)
        self.merges = []
        self.itos = list(BPE_SPECIALS)
        self.stoi = {t:i for i,t in enumerate(self.itos)}
        self.symbol_bytes = {}
        for b in range(256):
            s = f"{b:02x}"
            self.symbol_bytes[s] = bytes([b])
            self.stoi[s] = len(self.itos)
            self.itos.append(s)
        self._reset_encode_cache()

    @property
    def pad_token_id(self): return self.stoi["<pad>"]
    @property
    def unk_token_id(self): return self.stoi["<unk>"]
    @property
    def bos_token_id(self): return self.stoi["<s>"]
    @property
    def eos_token_id(self): return self.stoi["</s>"]
    def __len__(self): return len(self.itos)

    @staticmethod
    def word_to_symbols(word):
        return [f"{b:02x}" for b in word.encode("utf-8")]

    def _reset_encode_cache(self):
        self._ranks = {(l,r): i for i,(l,r) in enumerate(self.merges)}
        self._word_cache = {}
        self._cache_nmerges = len(self.merges)

    def fit(self, sentences, min_count=2):
        word_counts = Counter()
        seen_chars = 0
        t0 = time.time()

        for sentence in sentences:
            seen_chars += len(sentence)
            word_counts.update(re.findall(r"\S+|\s+", sentence))
            if seen_chars and seen_chars % 10_000_000 < len(sentence):
                print(f"  BPE corpus read: {seen_chars/1e6:.0f}M chars | unique units={len(word_counts):,}", flush=True)

        if min_count > 1:
            word_counts = Counter({w:c for w,c in word_counts.items() if c >= min_count})

        print(f"BPE training units after min_count={min_count}: {len(word_counts):,}")
        print(f"Counting time: {(time.time()-t0)/60:.1f} min")

        words = list(word_counts)
        freqs = [word_counts[w] for w in words]
        syms = [self.word_to_symbols(w) for w in words]

        target_merges = self.vocab_size - len(self.itos)
        pair_counts = Counter()
        pair_words = {}

        for wi, s in enumerate(syms):
            f = freqs[wi]
            for j in range(len(s)-1):
                p = (s[j], s[j+1])
                pair_counts[p] += f
                pair_words.setdefault(p, set()).add(wi)

        heap = [(-c,p) for p,c in pair_counts.items()]
        heapq.heapify(heap)

        for merge_num in range(target_merges):
            best_pair = None
            best_freq = 0
            while heap:
                negc, p = heap[0]
                if pair_counts.get(p,0) == -negc and -negc > 0:
                    best_pair, best_freq = p, -negc
                    break
                heapq.heappop(heap)

            if best_pair is None or best_freq <= 1:
                print("No useful pairs remain.")
                break

            left, right = best_pair
            new_symbol = left + right
            self.merges.append([left,right])
            self.symbol_bytes[new_symbol] = self.symbol_bytes[left] + self.symbol_bytes[right]
            if new_symbol not in self.stoi:
                self.stoi[new_symbol] = len(self.itos)
                self.itos.append(new_symbol)

            touched = set()
            for wi in list(pair_words.get(best_pair, ())):
                old = syms[wi]
                f = freqs[wi]

                # Remove old local pair contributions.
                for j in range(len(old)-1):
                    p = (old[j], old[j+1])
                    pair_counts[p] -= f
                    touched.add(p)

                merged = []
                j = 0
                while j < len(old):
                    if j+1 < len(old) and old[j] == left and old[j+1] == right:
                        merged.append(new_symbol)
                        j += 2
                    else:
                        merged.append(old[j])
                        j += 1
                syms[wi] = merged

                # Add new local pair contributions.
                for j in range(len(merged)-1):
                    p = (merged[j], merged[j+1])
                    pair_counts[p] += f
                    pair_words.setdefault(p, set()).add(wi)
                    touched.add(p)

            pair_words.pop(best_pair, None)

            for p in touched:
                c = pair_counts.get(p,0)
                if c > 0:
                    heapq.heappush(heap, (-c,p))
                else:
                    pair_counts.pop(p, None)

            if (merge_num+1) % 1000 == 0 or merge_num == 0 or merge_num+1 == target_merges:
                print(
                    f"merge {merge_num+1:,}/{target_merges:,} | "
                    f"freq={best_freq:,} | vocab={len(self.itos):,}",
                    flush=True
                )

        self.vocab_size = len(self.itos)
        self._reset_encode_cache()
        print(f"BPE complete: vocab={len(self):,}")
        return self

    def encode_word(self, word):
        if self._cache_nmerges != len(self.merges):
            self._reset_encode_cache()

        hit = self._word_cache.get(word)
        if hit is not None:
            return list(hit)

        symbols = self.word_to_symbols(word)
        ranks = self._ranks

        while len(symbols) > 1:
            best = None
            best_rank = None
            for j in range(len(symbols)-1):
                r = ranks.get((symbols[j], symbols[j+1]))
                if r is not None and (best_rank is None or r < best_rank):
                    best = (symbols[j], symbols[j+1])
                    best_rank = r
            if best is None:
                break

            left, right = best
            merged = []
            j = 0
            while j < len(symbols):
                if j+1 < len(symbols) and symbols[j] == left and symbols[j+1] == right:
                    merged.append(left+right)
                    j += 2
                else:
                    merged.append(symbols[j])
                    j += 1
            symbols = merged

        # Bound cache memory for 500M-token preprocessing.
        if len(self._word_cache) >= 250_000:
            self._word_cache.clear()
        self._word_cache[word] = tuple(symbols)
        return symbols

    def encode(self, text):
        out = []
        unk = self.unk_token_id
        for unit in re.findall(r"\S+|\s+", text):
            for s in self.encode_word(unit):
                out.append(self.stoi.get(s, unk))
        return out

    def decode(self, ids):
        buf = bytearray()
        for idx in ids:
            idx = int(idx)
            if idx < 0 or idx >= len(self.itos):
                raise ValueError(f"invalid token id {idx}")
            tok = self.itos[idx]
            if tok in BPE_SPECIALS:
                continue
            buf.extend(self.symbol_bytes[tok])
        return bytes(buf).decode("utf-8", errors="replace")

    def save_json(self, path):
        with open(path, "w", encoding="utf-8") as f:
            json.dump(
                {"vocab_size":len(self), "merges":self.merges, "itos":self.itos},
                f, ensure_ascii=False
            )

    @classmethod
    def load_json(cls, path):
        obj = json.load(open(path, encoding="utf-8"))
        tok = cls(obj["vocab_size"])
        tok.merges = [list(x) for x in obj["merges"]]
        tok.itos = list(obj["itos"])
        tok.stoi = {t:i for i,t in enumerate(tok.itos)}
        tok.vocab_size = len(tok.itos)

        tok.symbol_bytes = {}
        for b in range(256):
            s=f"{b:02x}"
            tok.symbol_bytes[s]=bytes([b])
        for l,r in tok.merges:
            tok.symbol_bytes[l+r] = tok.symbol_bytes[l] + tok.symbol_bytes[r]

        tok._reset_encode_cache()
        return tok

print("Tokenizer class ready.")

tokenizer = LegalBPE.load_json(TOKENIZER_FILE)
print("Tokenizer vocab:",len(tokenizer))
print("special ids:",
      tokenizer.pad_token_id,tokenizer.unk_token_id,
      tokenizer.bos_token_id,tokenizer.eos_token_id)
assert len(tokenizer)==VOCAB_SIZE
assert tokenizer.pad_token_id==0
print("✅ tokenizer sanity PASS")


Tokenizer class ready.
Tokenizer vocab: 24576
special ids: 0 1 2 3
✅ tokenizer sanity PASS


In [7]:
# CELL 4 — authenticate/download official Phase-5 data
HF_TOKEN=None
try:
    from kaggle_secrets import UserSecretsClient
    HF_TOKEN=UserSecretsClient().get_secret("HF_TOKEN")
except Exception as e:
    print("HF_TOKEN secret unavailable:",repr(e)[:120])

if HF_TOKEN:
    login(token=HF_TOKEN,add_to_git_credential=False)

HF_REPO="Exploration-Lab/CS-781-Capstone"
FNS={
    "train":"Phase_5_LSI/Train/train.json",
    "dev":"Phase_5_LSI/Train/dev.json",
    "test":"Phase_5_LSI/Test/test.json",
}
paths={
    k:hf_hub_download(repo_id=HF_REPO,repo_type="dataset",filename=v,token=HF_TOKEN)
    for k,v in FNS.items()
}
DATA_SHA256={k:sha256_file(v) for k,v in paths.items()}

def read_json(path):
    with open(path,encoding="utf-8") as f: return json.load(f)

train_raw=read_json(paths["train"])
dev_raw=read_json(paths["dev"])
test_raw=read_json(paths["test"])

print("rows:",len(train_raw),len(dev_raw),len(test_raw))
assert len(train_raw)==42750 and len(dev_raw)==10181 and len(test_raw)==13019

for name,recs,labeled in [("TRAIN",train_raw,True),("DEV",dev_raw,True),("TEST",test_raw,False)]:
    ids=[str(r["id"]) for r in recs]
    assert len(ids)==len(set(ids)),f"{name} duplicate IDs"
    for r in recs:
        assert isinstance(r.get("text"),str)
        if labeled:
            assert isinstance(r.get("labels"),list)
            r["labels"]=sorted(set(int(x) for x in r["labels"]))
            assert all(0<=x<N_LABELS for x in r["labels"])

assert not any("labels" in r for r in test_raw)
print("✅ schema/ID/label sanity PASS")


Phase_5_LSI/Train/train.json: reconstructing file:   0%|          |  0.00B /  299MB            

Phase_5_LSI/Train/train.json: downloading bytes:           |  0.00B            

Phase_5_LSI/Train/dev.json: reconstructing file:   0%|          |  0.00B / 79.4MB            

Phase_5_LSI/Train/dev.json: downloading bytes:           |  0.00B            

Phase_5_LSI/Test/test.json: reconstructing file:   0%|          |  0.00B / 98.1MB            

Phase_5_LSI/Test/test.json: downloading bytes:           |  0.00B            

rows: 42750 10181 13019
✅ schema/ID/label sanity PASS


In [8]:
# CELL 5 — leakage-aware clean DEV + class weights
train_hashes={text_hash(r["text"]) for r in train_raw}
dev_hashes={text_hash(r["text"]) for r in dev_raw}
test_hashes={text_hash(r["text"]) for r in test_raw}

print("text overlap train-dev:",len(train_hashes&dev_hashes))
print("text overlap train-test:",len(train_hashes&test_hashes))
print("text overlap dev-test:",len(dev_hashes&test_hashes))

CLEAN_IDX=np.asarray(
    [i for i,r in enumerate(dev_raw) if text_hash(r["text"]) not in train_hashes],
    dtype=np.int64
)
clean_dev=[dev_raw[i] for i in CLEAN_IDX]
train_records=train_raw
test_records=test_raw

print("official DEV:",len(dev_raw),"clean DEV:",len(clean_dev),"removed:",len(dev_raw)-len(clean_dev))
assert len(clean_dev)>9000

train_counts=np.zeros(N_LABELS,dtype=np.int64)
for r in train_records: train_counts[r["labels"]]+=1

dev_counts=np.zeros(N_LABELS,dtype=np.int64)
for r in clean_dev: dev_counts[r["labels"]]+=1

assert np.all(train_counts>0)
assert np.all(dev_counts>0)

raw_pw=(len(train_records)-train_counts)/train_counts
POS_WEIGHT=np.clip(np.sqrt(raw_pw),1.0,10.0).astype(np.float32)
print("pos_weight min/median/max:",POS_WEIGHT.min(),np.median(POS_WEIGHT),POS_WEIGHT.max())

official_truth=[{"id":str(r["id"]),"gold":[int(x) for x in r["labels"]]} for r in dev_raw]
clean_truth=[{"id":str(r["id"]),"gold":[int(x) for x in r["labels"]]} for r in clean_dev]
json.dump(official_truth,open(ROOT/"dev_truth_official.json","w"))
json.dump(clean_truth,open(ROOT/"dev_truth_clean.json","w"))


text overlap train-dev: 81
text overlap train-test: 109
text overlap dev-test: 62
official DEV: 10181 clean DEV: 10063 removed: 118
pos_weight min/median/max: 1.6261922 7.871644 10.0


In [9]:
# CELL 6 — fast 3-view statute-aware token cache
CUE_RE=re.compile(
    r"\b(section|sections|sec\.?|article|articles|act|acts|rule|rules|"
    r"provision|provisions|ipc|crpc|constitution)\b",re.I
)

def _char_window(text,center,width=CHAR_WINDOW):
    if len(text)<=width: return text
    st=max(0,min(len(text)-width,int(center-width//2)))
    return text[st:st+width]

def select_views(text):
    n=len(text)
    if n<=CHAR_WINDOW:
        return [text,text,text]

    head=text[:CHAR_WINDOW]
    tail=text[-CHAR_WINDOW:]

    hits=[m.start() for m in CUE_RE.finditer(text)]
    if hits:
        best_i=best_j=0; j=0
        for i in range(len(hits)):
            if j<i: j=i
            while j+1<len(hits) and hits[j+1]-hits[i]<=CHAR_WINDOW:
                j+=1
            if j-i>best_j-best_i:
                best_i,best_j=i,j
        center=(hits[best_i]+hits[best_j])//2
        evid=_char_window(text,center)
    else:
        evid=_char_window(text,n//2)

    return [head,evid,tail]

def encode_view(text,view_idx):
    ids=tokenizer.encode(text)
    if not ids: ids=[tokenizer.unk_token_id]
    if len(ids)>CHUNK_LEN:
        if view_idx==0:
            ids=ids[:CHUNK_LEN]
        elif view_idx==2:
            ids=ids[-CHUNK_LEN:]
        else:
            st=(len(ids)-CHUNK_LEN)//2
            ids=ids[st:st+CHUNK_LEN]
    return ids

def cache_split(name,records):
    ids_path=CACHE_DIR/f"{name}_ids.npy"
    len_path=CACHE_DIR/f"{name}_lens.npy"
    meta_path=CACHE_DIR/f"{name}_meta.json"

    dataset_key="dev" if name.startswith("dev") else name
    fingerprint={
        "dataset_sha256":DATA_SHA256[dataset_key],
        "tokenizer_sha256":sha256_file(TOKENIZER_FILE),
        "n":len(records),
        "n_chunks":N_CHUNKS,
        "chunk_len":CHUNK_LEN,
        "char_window":CHAR_WINDOW,
        "views":VIEW_NAMES,
        "ids_hash":hashlib.sha256("|".join(str(r["id"]) for r in records).encode()).hexdigest(),
    }

    if ids_path.exists() and len_path.exists() and meta_path.exists():
        if json.load(open(meta_path))==fingerprint:
            print("✅ reuse token cache:",name)
            return np.load(ids_path,mmap_mode="r"),np.load(len_path,mmap_mode="r")

    arr=np.full((len(records),N_CHUNKS,CHUNK_LEN),tokenizer.pad_token_id,dtype=np.uint16)
    lens=np.zeros((len(records),N_CHUNKS),dtype=np.uint16)

    t0=time.time()
    for i,r in enumerate(records):
        for c,txt in enumerate(select_views(r["text"])):
            z=encode_view(txt,c)
            L=min(len(z),CHUNK_LEN)
            arr[i,c,:L]=np.asarray(z[:L],dtype=np.uint16)
            lens[i,c]=L
        if (i+1)%2500==0 or i+1==len(records):
            print(f"{name}: {i+1}/{len(records)} elapsed={(time.time()-t0)/60:.1f}m",flush=True)

    np.save(ids_path,arr); np.save(len_path,lens)
    json.dump(fingerprint,open(meta_path,"w"),indent=2)
    del arr,lens; gc.collect()
    return np.load(ids_path,mmap_mode="r"),np.load(len_path,mmap_mode="r")

train_tok,train_lens=cache_split("train",train_records)
official_tok,official_lens=cache_split("dev",dev_raw)
test_tok,test_lens=cache_split("test",test_records)

print("cache shapes:",train_tok.shape,official_tok.shape,test_tok.shape)
assert train_tok.shape==(len(train_records),N_CHUNKS,CHUNK_LEN)


train: 2500/42750 elapsed=0.1m
train: 5000/42750 elapsed=0.3m
train: 7500/42750 elapsed=0.4m
train: 10000/42750 elapsed=0.5m
train: 12500/42750 elapsed=0.6m
train: 15000/42750 elapsed=0.8m
train: 17500/42750 elapsed=0.9m
train: 20000/42750 elapsed=1.0m
train: 22500/42750 elapsed=1.1m
train: 25000/42750 elapsed=1.2m
train: 27500/42750 elapsed=1.4m
train: 30000/42750 elapsed=1.5m
train: 32500/42750 elapsed=1.6m
train: 35000/42750 elapsed=1.7m
train: 37500/42750 elapsed=1.8m
train: 40000/42750 elapsed=2.0m
train: 42500/42750 elapsed=2.1m
train: 42750/42750 elapsed=2.1m
dev: 2500/10181 elapsed=0.1m
dev: 5000/10181 elapsed=0.3m
dev: 7500/10181 elapsed=0.4m
dev: 10000/10181 elapsed=0.5m
dev: 10181/10181 elapsed=0.5m
test: 2500/13019 elapsed=0.1m
test: 5000/13019 elapsed=0.3m
test: 7500/13019 elapsed=0.4m
test: 10000/13019 elapsed=0.5m
test: 12500/13019 elapsed=0.7m
test: 13019/13019 elapsed=0.7m
cache shapes: (42750, 3, 320) (10181, 3, 320) (13019, 3, 320)


In [10]:
# CELL 7 — exact Phase-1 Model2 decoder-only architecture
# Cell 9 — ~80M Decoder-only causal Transformer with SDPA + KV cache
class SinusoidalPositionalEncoding(nn.Module):
    def __init__(self, d_model, max_positions):
        super().__init__()
        pos = torch.arange(max_positions, dtype=torch.float32).unsqueeze(1)
        div = torch.exp(
            torch.arange(0, d_model, 2, dtype=torch.float32)
            * (-math.log(10000.0) / d_model)
        )
        pe = torch.zeros(max_positions, d_model, dtype=torch.float32)
        pe[:, 0::2] = torch.sin(pos * div)
        pe[:, 1::2] = torch.cos(pos * div)
        self.register_buffer("pe", pe, persistent=False)

    def forward(self, position_ids):
        return self.pe[position_ids]


class CausalSelfAttention(nn.Module):
    def __init__(self, d_model, n_heads, dropout):
        super().__init__()
        assert d_model % n_heads == 0
        self.n_heads = n_heads
        self.head_dim = d_model // n_heads
        self.qkv = nn.Linear(d_model, 3 * d_model)
        self.out_proj = nn.Linear(d_model, d_model)
        self.dropout = dropout

    def forward(self, x, past_kv=None, use_cache=False, key_valid=None):
        B, T, C = x.shape

        qkv = self.qkv(x).view(B, T, 3, self.n_heads, self.head_dim)
        q, k, v = qkv.unbind(dim=2)
        q = q.transpose(1, 2)
        k = k.transpose(1, 2)
        v = v.transpose(1, 2)

        past_len = 0
        if past_kv is not None:
            pk, pv = past_kv
            past_len = pk.size(2)
            k = torch.cat([pk, k], dim=2)
            v = torch.cat([pv, v], dim=2)

        # Training / ordinary prefill: use SDPA's efficient causal path.
        # Left-padded batched prefill needs an explicit causal + key-valid mask.
        # Cached one-token decoding already has only past/current keys, so a key-valid mask is enough.
        attn_mask = None
        is_causal = False

        if key_valid is not None:
            key_mask = key_valid[:, None, None, :]  # (B,1,1,S), True = real key
            if past_len == 0 and T > 1:
                causal = torch.ones((T, T), dtype=torch.bool, device=x.device).tril()
                attn_mask = key_mask & causal[None, None, :, :]

                # Left-padding creates early PAD query rows with no valid causal key.
                # Giving each such row access to itself avoids fully-masked SDPA rows
                # on kernels/versions that may otherwise produce NaNs. Real-token
                # attention is unchanged because real positions already include self.
                eye = torch.eye(T, dtype=torch.bool, device=x.device)[None, None, :, :]
                attn_mask = attn_mask | eye
            else:
                attn_mask = key_mask
        elif past_kv is None:
            is_causal = True

        y = F.scaled_dot_product_attention(
            q, k, v,
            attn_mask=attn_mask,
            dropout_p=self.dropout if self.training else 0.0,
            is_causal=is_causal,
        )

        y = y.transpose(1, 2).contiguous().view(B, T, C)
        y = self.out_proj(y)

        new_kv = (k, v) if use_cache else None
        return y, new_kv


class DecoderBlock(nn.Module):
    def __init__(self, d_model, n_heads, ffn_dim, dropout):
        super().__init__()
        self.ln1 = nn.LayerNorm(d_model)
        self.attn = CausalSelfAttention(d_model, n_heads, dropout)
        self.ln2 = nn.LayerNorm(d_model)
        self.ff = nn.Sequential(
            nn.Linear(d_model, ffn_dim),
            nn.GELU(),
            nn.Linear(ffn_dim, d_model),
            nn.Dropout(dropout),
        )
        self.resid_drop = nn.Dropout(dropout)

    def forward(self, x, past_kv=None, use_cache=False, key_valid=None):
        a, new_kv = self.attn(
            self.ln1(x),
            past_kv=past_kv,
            use_cache=use_cache,
            key_valid=key_valid,
        )
        x = x + self.resid_drop(a)
        x = x + self.ff(self.ln2(x))
        return x, new_kv


class DecoderOnlyLM(nn.Module):
    def __init__(self, vocab_size, d_model, n_layers, n_heads, ffn_dim, dropout, max_positions):
        super().__init__()
        self.vocab_size = vocab_size
        self.d_model = d_model
        self.max_positions = max_positions

        self.token_emb = nn.Embedding(vocab_size, d_model)
        self.pos = SinusoidalPositionalEncoding(d_model, max_positions)
        self.drop = nn.Dropout(dropout)

        self.blocks = nn.ModuleList([
            DecoderBlock(d_model, n_heads, ffn_dim, dropout)
            for _ in range(n_layers)
        ])
        self.final_ln = nn.LayerNorm(d_model)

        self.apply(self._init_weights)

        # GPT-style scaled residual projections.
        scaled_std = 0.02 / math.sqrt(2 * n_layers)
        for block in self.blocks:
            nn.init.normal_(block.attn.out_proj.weight, mean=0.0, std=scaled_std)
            nn.init.normal_(block.ff[2].weight, mean=0.0, std=scaled_std)

    def _init_weights(self, m):
        if isinstance(m, nn.Linear):
            nn.init.normal_(m.weight, mean=0.0, std=0.02)
            if m.bias is not None:
                nn.init.zeros_(m.bias)
        elif isinstance(m, nn.Embedding):
            nn.init.normal_(m.weight, mean=0.0, std=0.02)

    def forward_hidden(
        self,
        input_ids,
        past_key_values=None,
        use_cache=False,
        position_ids=None,
        key_valid=None,
    ):
        B, T = input_ids.shape

        past_len = 0
        if past_key_values is not None:
            past_len = past_key_values[0][0].size(2)

        if past_len + T > self.max_positions:
            raise ValueError(
                f"context {past_len + T} exceeds max_positions={self.max_positions}"
            )

        if position_ids is None:
            position_ids = torch.arange(
                past_len, past_len + T, device=input_ids.device
            )[None, :].expand(B, T)

        x = self.token_emb(input_ids) + self.pos(position_ids).to(self.token_emb.weight.dtype)
        x = self.drop(x)

        new_cache = [] if use_cache else None

        for i, block in enumerate(self.blocks):
            past = None if past_key_values is None else past_key_values[i]
            x, kv = block(
                x,
                past_kv=past,
                use_cache=use_cache,
                key_valid=key_valid,
            )
            if use_cache:
                new_cache.append(kv)

        x = self.final_ln(x)
        return x, new_cache

    def project(self, hidden):
        # Tied token/output embedding.
        return F.linear(hidden, self.token_emb.weight)


In [11]:
# CELL 8 — load Model2 + inject SVD-style AdaLoRA adapters
def build_backbone():
    return DecoderOnlyLM(
        vocab_size=len(tokenizer),
        d_model=D_MODEL,
        n_layers=N_LAYERS,
        n_heads=N_HEADS,
        ffn_dim=FFN_DIM,
        dropout=DROPOUT,
        max_positions=MAX_POSITIONS,
    )

backbone=build_backbone()
phase1_obj=torch.load(MODEL2_CKPT,map_location="cpu",weights_only=False)
assert isinstance(phase1_obj,dict) and "model" in phase1_obj

missing,unexpected=backbone.load_state_dict(phase1_obj["model"],strict=False)
assert not missing and not unexpected,(missing,unexpected)
print(
    "✅ Model2 load PASS | %.3fM params"
    %(sum(p.numel() for p in backbone.parameters())/1e6)
)
print("checkpoint global_step:",phase1_obj.get("global_step"))
print("checkpoint best_val:",phase1_obj.get("best_val"))
print("checkpoint val_loss:",phase1_obj.get("val_loss"))

for p in backbone.parameters():
    p.requires_grad=False


class AdaLoRALinear(nn.Module):
    """
    Frozen W0 + SVD-style adaptive low-rank update:

        y = W0 x + scale * P diag(E * mask) Q x

    P: [out, r], E: [r], Q: [r, in]

    E starts at zero -> exact pretrained function at initialization.
    rank_mask is changed by a GLOBAL AdaLoRA budget allocator.
    """
    def __init__(
        self,base_linear,rank=12,alpha=16.0,init_std=0.02,
        beta1=0.85,beta2=0.85
    ):
        super().__init__()
        assert isinstance(base_linear,nn.Linear)
        self.base=base_linear
        for p in self.base.parameters():
            p.requires_grad=False

        self.rank=int(rank)
        self.alpha=float(alpha)
        self.scaling=self.alpha/self.rank
        self.beta1=float(beta1)
        self.beta2=float(beta2)

        din=base_linear.in_features
        dout=base_linear.out_features

        self.P=nn.Parameter(torch.empty(dout,self.rank))
        self.E=nn.Parameter(torch.zeros(self.rank))
        self.Q=nn.Parameter(torch.empty(self.rank,din))

        nn.init.normal_(self.P,mean=0.0,std=init_std)
        nn.init.normal_(self.Q,mean=0.0,std=init_std)

        self.register_buffer(
            "rank_mask",torch.ones(self.rank,dtype=torch.float32)
        )
        self.register_buffer(
            "ipt_ema",torch.zeros(self.rank,dtype=torch.float32)
        )
        self.register_buffer(
            "unc_ema",torch.zeros(self.rank,dtype=torch.float32)
        )

    def forward(self,x):
        y0=self.base(x)
        e=self.E*self.rank_mask.to(self.E.dtype)
        z=F.linear(x,self.Q)
        z=z*e
        delta=F.linear(z,self.P)
        return y0+self.scaling*delta

    @torch.no_grad()
    def update_importance(self):
        # Triplet-level sensitivity: all three SVD-like factors contribute.
        pieces=[]
        if self.E.grad is not None:
            pieces.append((self.E*self.E.grad).abs())
        if self.P.grad is not None:
            pieces.append((self.P*self.P.grad).abs().mean(0))
        if self.Q.grad is not None:
            pieces.append((self.Q*self.Q.grad).abs().mean(1))

        if not pieces:
            return

        sens=torch.stack(pieces).mean(0).float()
        old=self.ipt_ema.clone()
        self.ipt_ema.mul_(self.beta1).add_(sens,alpha=1.0-self.beta1)
        dev=(sens-old).abs()
        self.unc_ema.mul_(self.beta2).add_(dev,alpha=1.0-self.beta2)

    @torch.no_grad()
    def importance_score(self):
        # Sensitivity × uncertainty, the core AdaLoRA ranking idea.
        return self.ipt_ema*self.unc_ema

    def orthogonality_loss(self):
        # Encourage approximately orthogonal singular directions.
        eye=torch.eye(self.rank,device=self.P.device,dtype=self.P.dtype)
        pgram=self.P.transpose(0,1)@self.P
        qgram=self.Q@self.Q.transpose(0,1)

        pgram=pgram/(pgram.diag().mean().detach().clamp_min(1e-6))
        qgram=qgram/(qgram.diag().mean().detach().clamp_min(1e-6))
        return ((pgram-eye)**2).mean()+((qgram-eye)**2).mean()


def inject_adalora(backbone):
    names=[]
    for bi in ADAPTED_BLOCKS:
        block=backbone.blocks[bi]

        block.attn.qkv=AdaLoRALinear(
            block.attn.qkv,ADALORA_INIT_RANK,ADALORA_ALPHA,
            ADALORA_INIT_STD,ADALORA_BETA1,ADALORA_BETA2
        )
        names.append(f"blocks.{bi}.attn.qkv")

        block.attn.out_proj=AdaLoRALinear(
            block.attn.out_proj,ADALORA_INIT_RANK,ADALORA_ALPHA,
            ADALORA_INIT_STD,ADALORA_BETA1,ADALORA_BETA2
        )
        names.append(f"blocks.{bi}.attn.out_proj")

        block.ff[0]=AdaLoRALinear(
            block.ff[0],ADALORA_INIT_RANK,ADALORA_ALPHA,
            ADALORA_INIT_STD,ADALORA_BETA1,ADALORA_BETA2
        )
        names.append(f"blocks.{bi}.ff.0")

        block.ff[2]=AdaLoRALinear(
            block.ff[2],ADALORA_INIT_RANK,ADALORA_ALPHA,
            ADALORA_INIT_STD,ADALORA_BETA1,ADALORA_BETA2
        )
        names.append(f"blocks.{bi}.ff.2")

    return names


def adalora_modules(model):
    return [m for m in model.modules() if isinstance(m,AdaLoRALinear)]


# Exact identity-at-initialization check.
backbone.eval()
with torch.inference_mode():
    probe=torch.tensor([[4,5,6,7,8,9]],dtype=torch.long)
    h0,_=backbone.forward_hidden(probe)

injected_modules=inject_adalora(backbone)

with torch.inference_mode():
    h1,_=backbone.forward_hidden(probe)

init_diff=(h0-h1).abs().max().item()
print("Injected AdaLoRA modules:",len(injected_modules))
print("max init hidden diff:",init_diff)
assert init_diff<1e-6

bad=[
    n for n,p in backbone.named_parameters()
    if p.requires_grad
    and not any(k in n for k in [".P",".E",".Q"])
]
assert not bad,bad

print(
    "AdaLoRA backbone trainable params:",
    sum(p.numel() for p in backbone.parameters() if p.requires_grad)
)


class AdaRankAllocator:
    """Global cubic rank-budget scheduler."""
    def __init__(
        self,model,total_steps,init_rank,target_avg_rank,
        warmup_frac=0.10,final_frac=0.20,interval=10
    ):
        self.modules=adalora_modules(model)
        assert self.modules
        self.total_steps=max(1,int(total_steps))
        self.interval=int(interval)

        self.init_budget=len(self.modules)*int(init_rank)
        self.target_budget=len(self.modules)*int(target_avg_rank)
        self.warmup_steps=max(1,int(warmup_frac*self.total_steps))
        self.final_steps=max(1,int(final_frac*self.total_steps))
        assert self.target_budget<=self.init_budget

    def budget_at(self,step):
        step=int(step)
        if step<=self.warmup_steps:
            return self.init_budget
        if step>=self.total_steps-self.final_steps:
            return self.target_budget

        span=max(
            1,self.total_steps-self.warmup_steps-self.final_steps
        )
        p=(step-self.warmup_steps)/span
        # AdaLoRA-style cubic decay.
        b=self.target_budget+(self.init_budget-self.target_budget)*(1.0-p)**3
        return int(round(b))

    @torch.no_grad()
    def update_importance(self):
        for m in self.modules:
            m.update_importance()

    @torch.no_grad()
    def reallocate(self,step,force=False):
        if (not force) and (
            step<=self.warmup_steps or step%self.interval!=0
        ):
            return self.active_budget()

        budget=self.budget_at(step)
        scores=torch.cat(
            [m.importance_score().detach().float().cpu() for m in self.modules]
        )
        budget=max(1,min(int(budget),scores.numel()))

        if budget==scores.numel():
            keep=torch.ones_like(scores,dtype=torch.bool)
        else:
            # Stable deterministic top-k. During an early tie this simply
            # keeps a valid subset; after warmup scores are normally distinct.
            idx=torch.topk(scores,budget,largest=True,sorted=False).indices
            keep=torch.zeros_like(scores,dtype=torch.bool)
            keep[idx]=True

        off=0
        for m in self.modules:
            r=m.rank
            m.rank_mask.copy_(
                keep[off:off+r].to(
                    device=m.rank_mask.device,dtype=m.rank_mask.dtype
                )
            )
            off+=r

        return budget

    def active_budget(self):
        return int(sum(m.rank_mask.sum().item() for m in self.modules))

    def rank_summary(self):
        ranks=[int(m.rank_mask.sum().item()) for m in self.modules]
        return {
            "min":int(min(ranks)),
            "median":float(np.median(ranks)),
            "max":int(max(ranks)),
            "total":int(sum(ranks)),
            "per_module":ranks,
        }


def adalora_orth_loss(model):
    mods=adalora_modules(model)
    if not mods:
        return torch.zeros((),device=DEVICE)
    return torch.stack([m.orthogonality_loss() for m in mods]).mean()


def get_adalora_mask_state(model):
    return {
        n:m.rank_mask.detach().cpu().clone()
        for n,m in model.named_modules()
        if isinstance(m,AdaLoRALinear)
    }


def load_adalora_mask_state(model,state):
    mods=dict(model.named_modules())
    with torch.no_grad():
        for n,v in state.items():
            assert n in mods and isinstance(mods[n],AdaLoRALinear),n
            mods[n].rank_mask.copy_(v.to(mods[n].rank_mask.device))


✅ Model2 load PASS | 79.736M params
checkpoint global_step: 16266
checkpoint best_val: None
checkpoint val_loss: 1.7356935616699047
Injected AdaLoRA modules: 28
max init hidden diff: 0.0
AdaLoRA backbone trainable params: 860496


In [12]:
# CELL 9 — lightweight 3-view Model2 AdaLoRA classifier
class StatuteAdaLoRAClassifier(nn.Module):
    def __init__(
        self,backbone,d_model=640,proj_dim=320,
        n_labels=100,dropout=0.10
    ):
        super().__init__()
        self.backbone=backbone

        self.proj=nn.Sequential(
            nn.Linear(2*d_model,proj_dim),
            nn.GELU(),
            nn.LayerNorm(proj_dim),
            nn.Dropout(dropout),
        )

        self.chunk_pos=nn.Parameter(torch.zeros(N_CHUNKS,proj_dim))
        nn.init.normal_(self.chunk_pos,std=0.02)

        self.attn=nn.Sequential(
            nn.Linear(proj_dim,96),
            nn.Tanh(),
            nn.Linear(96,1),
        )

        self.head=nn.Sequential(
            nn.Linear(2*proj_dim,proj_dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(proj_dim,n_labels),
        )

    def encode_flat_chunks(self,ids,valid):
        B,T=ids.shape
        position_ids=torch.arange(
            T,device=ids.device
        )[None,:].expand(B,T)

        # Right padding permits causal SDPA without an explicit key mask.
        kv=None if FAST_CAUSAL_NO_MASK else valid

        # Entire lower prefix is frozen.
        with torch.no_grad():
            x=(
                self.backbone.token_emb(ids)
                +self.backbone.pos(position_ids).to(
                    self.backbone.token_emb.weight.dtype
                )
            )
            x=self.backbone.drop(x)
            for block in self.backbone.blocks[:ADAPT_START_BLOCK]:
                x,_=block(
                    x,past_kv=None,use_cache=False,key_valid=kv
                )
        x=x.detach()

        # Top blocks contain trainable AdaLoRA triplets.
        for block in self.backbone.blocks[ADAPT_START_BLOCK:]:
            x,_=block(
                x,past_kv=None,use_cache=False,key_valid=kv
            )

        x=self.backbone.final_ln(x)

        mask=valid.unsqueeze(-1).to(x.dtype)
        mean=(x*mask).sum(1)/mask.sum(1).clamp_min(1.0)
        last_idx=valid.long().sum(1).clamp_min(1)-1
        last=x[torch.arange(B,device=x.device),last_idx]

        return self.proj(torch.cat([last,mean],dim=-1))

    def forward(self,input_ids,valid):
        B,C,T=input_ids.shape
        z=self.encode_flat_chunks(
            input_ids.reshape(B*C,T),
            valid.reshape(B*C,T)
        ).reshape(B,C,-1)

        z=z+self.chunk_pos[None,:,:]
        scores=self.attn(z).squeeze(-1)
        weights=torch.softmax(scores,dim=-1)

        attn_pool=(z*weights.unsqueeze(-1)).sum(1)
        max_pool=z.max(1).values
        logits=self.head(torch.cat([attn_pool,max_pool],dim=-1))
        return logits,weights


model=StatuteAdaLoRAClassifier(backbone).to(DEVICE)

total=sum(p.numel() for p in model.parameters())
trainable=sum(p.numel() for p in model.parameters() if p.requires_grad)
backbone_trainable=sum(
    p.numel() for p in model.backbone.parameters() if p.requires_grad
)

print(f"total={total/1e6:.3f}M")
print(f"trainable={trainable/1e6:.3f}M ({100*trainable/total:.3f}%)")
print(f"AdaLoRA backbone trainable={backbone_trainable/1e6:.3f}M")
assert 0<trainable<total and backbone_trainable>0


total=81.276M
trainable=1.540M (1.895%)
AdaLoRA backbone trainable=0.860M


In [13]:
# CELL 10 — datasets/loaders
class LSIDataset(Dataset):
    def __init__(self,records,tok,lens,labeled=True):
        self.records=records; self.tok=tok; self.lens=lens; self.labeled=labeled
    def __len__(self): return len(self.records)
    def __getitem__(self,i):
        ids=torch.from_numpy(np.array(self.tok[i],dtype=np.int64,copy=True))
        lens=torch.from_numpy(np.array(self.lens[i],dtype=np.int64,copy=True))
        valid=torch.arange(CHUNK_LEN)[None,:]<lens[:,None]
        out={"input_ids":ids,"valid":valid,"id":str(self.records[i]["id"])}
        if self.labeled:
            y=torch.zeros(N_LABELS,dtype=torch.float32)
            y[self.records[i]["labels"]]=1.0
            out["labels"]=y
        return out

def make_loader(records,tok,lens,batch,shuffle,labeled=True,seed=SEED):
    # v2: persistent workers only for the training loader (dev/test loaders are rebuilt cheaply & survive interrupts)
    g=torch.Generator(); g.manual_seed(seed)
    return DataLoader(
        LSIDataset(records,tok,lens,labeled),
        batch_size=batch,shuffle=shuffle,num_workers=NUM_WORKERS,
        pin_memory=True,persistent_workers=(NUM_WORKERS>0 and shuffle),
        generator=g
    )

train_loader=make_loader(train_records,train_tok,train_lens,MICRO_BATCH_SIZE,True,True)
official_loader=make_loader(dev_raw,official_tok,official_lens,DEV_BATCH_SIZE,False,True)
test_loader=make_loader(test_records,test_tok,test_lens,DEV_BATCH_SIZE,False,False)

print("loader batches:",len(train_loader),len(official_loader),len(test_loader))
print("train tokens/microbatch:",MICRO_BATCH_SIZE*N_CHUNKS*CHUNK_LEN)


loader batches: 10688 637 814
train tokens/microbatch: 3840


In [14]:
# CELL 11 — exact evaluator logic + threshold tuning
def multilabel_metrics(y_true,y_pred):
    return {
        "macro_f1":float(f1_score(y_true,y_pred,average="macro",zero_division=0)),
        "micro_f1":float(f1_score(y_true,y_pred,average="micro",zero_division=0)),
        "subset_accuracy":float(accuracy_score(y_true,y_pred)),
    }

def apply_thresholds(prob,thresholds,ensure_nonempty=True):
    thr=np.asarray(thresholds,dtype=np.float32)
    pred=prob>=thr[None,:]
    if ensure_nonempty:
        empty=np.flatnonzero(~pred.any(1))
        if len(empty):
            best=np.argmax(prob[empty]/np.maximum(thr[None,:],1e-6),axis=1)
            pred[empty,best]=True
    return pred

def macro_f1_fast(y,pred):
    yb=y.astype(bool); pb=pred.astype(bool)
    tp=(yb&pb).sum(0).astype(np.float64)
    den=yb.sum(0)+pb.sum(0)
    return float(np.where(den>0,2*tp/np.maximum(den,1),0.0).mean())

def f1_curves(y,prob,grid=THR_GRID):
    yb=y.astype(bool); P=yb.sum(0).astype(np.float64)
    out=np.zeros((len(grid),y.shape[1]),dtype=np.float64)
    for gi,t in enumerate(grid):
        pr=prob>=t
        tp=(pr&yb).sum(0); den=pr.sum(0)+P
        out[gi]=np.where(den>0,2*tp/np.maximum(den,1),0.0)
    return out

def smooth_curves(c):
    p=np.pad(c,((1,1),(0,0)),mode="edge")
    return (p[:-2]+p[1:-1]+p[2:])/3.0

def fit_threshold_strategies(y,prob):
    c=f1_curves(y,prob)
    g=float(THR_GRID[int(np.argmax(c.mean(1)))])
    per=THR_GRID[np.argmax(smooth_curves(c),axis=0)].astype(np.float32)
    P=y.sum(0).astype(np.float64)
    out={"global":np.full(y.shape[1],g,dtype=np.float32),"per_label":per}
    for tau in SHRINK_TAUS:
        w=P/(P+tau)
        out[f"shrunk_tau_{int(tau)}"]=(w*per+(1-w)*g).astype(np.float32)
    return out,g,per

def optimize_thresholds(y,prob):
    full,g,per=fit_threshold_strategies(y,prob)
    n=len(y); cv={k:[] for k in full}
    for rep_i in range(CV_REPEATS):
        perm=np.random.RandomState(100+rep_i).permutation(n)
        a,b=perm[:n//2],perm[n//2:]
        for tr,te in ((a,b),(b,a)):
            strat,_,_=fit_threshold_strategies(y[tr],prob[tr])
            for k,thr in strat.items():
                cv[k].append(
                    macro_f1_fast(y[te],apply_thresholds(prob[te],thr,True))
                )
    cv={k:float(np.mean(v)) for k,v in cv.items()}
    name=max(cv,key=cv.get); thr=full[name]
    return {
        "name":name,"thresholds":thr,"cv_macro_f1":cv[name],
        "insample_macro_f1":macro_f1_fast(y,apply_thresholds(prob,thr,True)),
        "cv_all":cv
    },g,per

@torch.inference_mode()
def predict_loader(model,loader):
    model.eval()
    ids=[]; probs=[]; ys=[]; atts=[]
    for b in loader:
        x=b["input_ids"].to(DEVICE,non_blocking=True)
        v=b["valid"].to(DEVICE,non_blocking=True)
        with torch.amp.autocast(
            "cuda",dtype=torch.float16,
            enabled=(DEVICE.type=="cuda" and USE_AMP)
        ):
            lg,a=model(x,v)
        assert torch.isfinite(lg).all()
        ids.extend(list(b["id"]))
        probs.append(torch.sigmoid(lg.float()).cpu().numpy())
        atts.append(a.float().cpu().numpy())
        if "labels" in b: ys.append(b["labels"].numpy().astype(np.uint8))
    return ids,np.concatenate(probs),(np.concatenate(ys) if ys else None),np.concatenate(atts)

def make_pred_records(ids,prob,thresholds,ensure_nonempty=True):
    binary=apply_thresholds(prob,thresholds,ensure_nonempty)
    return [{"id":str(i),"pred":[int(x) for x in np.flatnonzero(binary[k])]} for k,i in enumerate(ids)]

def exact_phase5_score(truth_records,pred_records):
    t={}; max_label=-1
    for r in truth_records:
        assert r["id"] not in t
        seen=set()
        for label in r["gold"]:
            assert isinstance(label,int) and not isinstance(label,bool) and label>=0 and label not in seen
            seen.add(label); max_label=max(max_label,label)
        t[r["id"]]=r["gold"]

    nlab=max_label+1
    p={}
    for r in pred_records:
        assert r["id"] not in p
        seen=set()
        for x in r["pred"]:
            assert isinstance(x,int) and not isinstance(x,bool) and 0<=x<nlab and x not in seen
            seen.add(x)
        p[r["id"]]=r["pred"]

    assert set(t)==set(p)
    yt=np.zeros((len(t),nlab),dtype=np.uint8)
    yp=np.zeros((len(t),nlab),dtype=np.uint8)
    for i,k in enumerate(t):
        for x in t[k]: yt[i,x]=1
        for x in p[k]: yp[i,x]=1
    return multilabel_metrics(yt,yp)

print("✅ metric/threshold utilities ready")


✅ metric/threshold utilities ready


In [15]:
# CELL 12 — AdaLoRA training: adaptive rank budget + ONE DEV pass/epoch
pos_weight_t=torch.tensor(
    POS_WEIGHT,dtype=torch.float32,device=DEVICE
)

adalora_params=[]
head_params=[]
for n,p in model.named_parameters():
    if not p.requires_grad:
        continue
    if n.startswith("backbone.") and any(
        k in n for k in [".P",".E",".Q"]
    ):
        adalora_params.append(p)
    else:
        head_params.append(p)

assert adalora_params and head_params
print("AdaLoRA trainable:",sum(p.numel() for p in adalora_params))
print("Head trainable   :",sum(p.numel() for p in head_params))

optimizer=torch.optim.AdamW(
    [
        {"params":adalora_params,"lr":ADALORA_LR,"weight_decay":0.0},
        {"params":head_params,"lr":HEAD_LR,"weight_decay":WEIGHT_DECAY},
    ],
    betas=(0.9,0.999),eps=1e-8
)

updates_per_epoch=math.ceil(len(train_loader)/GRAD_ACCUM_STEPS)
total_updates=updates_per_epoch*EPOCHS
warmup=max(1,int(WARMUP_RATIO*total_updates))

def lr_lambda(step):
    if step<warmup:
        return (step+1)/warmup
    q=(step-warmup)/max(1,total_updates-warmup)
    return max(
        0.08,0.5*(1+math.cos(math.pi*min(1.0,q)))
    )

scheduler=torch.optim.lr_scheduler.LambdaLR(
    optimizer,lr_lambda
)
rank_allocator=AdaRankAllocator(
    model=model,
    total_steps=total_updates,
    init_rank=ADALORA_INIT_RANK,
    target_avg_rank=ADALORA_TARGET_AVG_RANK,
    warmup_frac=ADALORA_WARMUP_FRAC,
    final_frac=ADALORA_FINAL_FRAC,
    interval=ADALORA_MASK_INTERVAL,
)

scaler=torch.amp.GradScaler(
    "cuda",enabled=(DEVICE.type=="cuda" and USE_AMP)
)

BEST=ROOT/"best_exp3_adalora.pt"
HISTORY=ROOT/"history.json"


class EMA:
    def __init__(self,model,decay):
        self.decay=decay
        self.params={
            n:p for n,p in model.named_parameters()
            if p.requires_grad
        }
        self.shadow={
            n:p.detach().clone() for n,p in self.params.items()
        }
        self.n=0

    @torch.no_grad()
    def update(self):
        self.n+=1
        d=min(self.decay,(1.0+self.n)/(10.0+self.n))
        for n,p in self.params.items():
            self.shadow[n].mul_(d).add_(p.detach(),alpha=1.0-d)

    @contextlib.contextmanager
    def applied(self):
        backup={
            n:p.detach().clone() for n,p in self.params.items()
        }
        with torch.no_grad():
            for n,p in self.params.items():
                p.copy_(self.shadow[n])
        try:
            yield
        finally:
            with torch.no_grad():
                for n,p in self.params.items():
                    p.copy_(backup[n])


def get_trainable_state(model):
    return {
        n:p.detach().cpu().clone()
        for n,p in model.named_parameters()
        if p.requires_grad
    }


def load_trainable_state(model,state):
    named=dict(model.named_parameters())
    with torch.no_grad():
        for n,v in state.items():
            assert n in named,n
            named[n].copy_(
                v.to(named[n].device,dtype=named[n].dtype)
            )


ema=EMA(model,EMA_DECAY)
best_score=-1.0
history=[]
global_update=0
optimizer.zero_grad(set_to_none=True)


def evaluate_epoch(epoch,tag="ema"):
    global best_score

    oids,oprob,oy,oatt=predict_loader(model,official_loader)
    cprob=oprob[CLEAN_IDX]
    cy=oy[CLEAN_IDX]
    cids=[oids[i] for i in CLEAN_IDX]
    catt=oatt[CLEAN_IDX]

    tune,g,per=optimize_thresholds(cy,cprob)

    clean_pred=make_pred_records(
        cids,cprob,tune["thresholds"],True
    )
    official_pred=make_pred_records(
        oids,oprob,tune["thresholds"],True
    )

    exact_clean=exact_phase5_score(clean_truth,clean_pred)
    exact_official=exact_phase5_score(
        official_truth,official_pred
    )
    rank_info=rank_allocator.rank_summary()

    row={
        "epoch":epoch,
        "weights":tag,
        "strategy":tune["name"],
        "global_threshold":g,
        "cv_macro_f1":tune["cv_macro_f1"],
        "insample_macro_f1":tune["insample_macro_f1"],
        "cv_all":tune["cv_all"],
        "clean":exact_clean,
        "official":exact_official,
        "mean_pred_labels_clean":float(
            np.mean([len(r["pred"]) for r in clean_pred])
        ),
        "mean_gold_labels_clean":float(cy.sum(1).mean()),
        "attention_mean_clean":catt.mean(0).tolist(),
        "rank_summary":rank_info,
    }

    history.append(row)
    json.dump(history,open(HISTORY,"w"),indent=2)

    print(
        f"[{tag}] epoch={epoch} strat={tune['name']} "
        f"CV-macroF1={tune['cv_macro_f1']:.4f} | "
        f"clean={exact_clean['macro_f1']:.4f}/"
        f"{exact_clean['micro_f1']:.4f}/"
        f"{exact_clean['subset_accuracy']:.4f} | "
        f"official-macro={exact_official['macro_f1']:.4f} | "
        f"rank total/min/med/max="
        f"{rank_info['total']}/{rank_info['min']}/"
        f"{rank_info['median']:.1f}/{rank_info['max']}",
        flush=True
    )

    if tune["cv_macro_f1"]>best_score:
        best_score=tune["cv_macro_f1"]

        atomic_torch_save(
            {
                "trainable_state":get_trainable_state(model),
                "adalora_masks":get_adalora_mask_state(model),
                "thresholds":np.asarray(
                    tune["thresholds"],dtype=np.float32
                ),
                "threshold_strategy":tune["name"],
                "global_threshold":g,
                "per_label_thresholds":np.asarray(
                    per,dtype=np.float32
                ),
                "cv_macro_f1":tune["cv_macro_f1"],
                "clean_metrics":exact_clean,
                "official_metrics":exact_official,
                "rank_summary":rank_info,
                "epoch":epoch,
                "weights_tag":tag,
                "model2_sha256":sha256_file(MODEL2_CKPT),
                "tokenizer_sha256":sha256_file(TOKENIZER_FILE),
                "dataset_sha256":DATA_SHA256,
                "config":{
                    "method":"AdaLoRA",
                    "init_rank":ADALORA_INIT_RANK,
                    "target_avg_rank":ADALORA_TARGET_AVG_RANK,
                    "alpha":ADALORA_ALPHA,
                    "adapt_start_block":ADAPT_START_BLOCK,
                    "targets":list(ADALORA_TARGETS),
                    "mask_interval":ADALORA_MASK_INTERVAL,
                    "warmup_frac":ADALORA_WARMUP_FRAC,
                    "final_frac":ADALORA_FINAL_FRAC,
                    "n_chunks":N_CHUNKS,
                    "chunk_len":CHUNK_LEN,
                    "views":VIEW_NAMES,
                    "adalora_lr":ADALORA_LR,
                    "head_lr":HEAD_LR,
                    "epochs":EPOCHS,
                    "pos_weight":"sqrt(raw) clipped [1,10]",
                },
            },
            BEST,
        )
        print("✅ NEW BEST:",round(best_score,5))


print("batches/epoch:",len(train_loader))
print("optimizer updates:",total_updates)
print(
    "AdaLoRA global rank budget:",
    rank_allocator.init_budget,"->",rank_allocator.target_budget
)
print(
    "OOM fallback: MICRO_BATCH_SIZE=2, GRAD_ACCUM_STEPS=8."
)

for epoch in range(1,EPOCHS+1):
    model.train()
    t0=time.time()
    running=0.0
    good=0

    for bi,b in enumerate(train_loader,1):
        x=b["input_ids"].to(DEVICE,non_blocking=True)
        v=b["valid"].to(DEVICE,non_blocking=True)
        y=b["labels"].to(DEVICE,non_blocking=True)

        with torch.amp.autocast(
            "cuda",dtype=torch.float16,
            enabled=(DEVICE.type=="cuda" and USE_AMP)
        ):
            logits,_=model(x,v)
            bce=F.binary_cross_entropy_with_logits(
                logits,y,pos_weight=pos_weight_t
            )
            orth=adalora_orth_loss(model)
            loss=bce+ADALORA_ORTH_COEF*orth
            scaled_loss=loss/GRAD_ACCUM_STEPS

        if torch.isfinite(loss):
            scaler.scale(scaled_loss).backward()
            good+=1
            running+=float(bce.detach())
        else:
            print("⚠️ non-finite loss batch",bi)

        if bi%GRAD_ACCUM_STEPS==0 or bi==len(train_loader):
            if good:
                scaler.unscale_(optimizer)

                # Importance is measured on unscaled gradients BEFORE step.
                rank_allocator.update_importance()

                torch.nn.utils.clip_grad_norm_(
                    [p for p in model.parameters() if p.requires_grad],
                    GRAD_CLIP
                )

                old_scale=float(scaler.get_scale())
                scaler.step(optimizer)
                scaler.update()
                optimizer.zero_grad(set_to_none=True)

                if float(scaler.get_scale())>=old_scale:
                    scheduler.step()
                    global_update+=1
                    ema.update()
                    rank_allocator.reallocate(global_update)
            else:
                optimizer.zero_grad(set_to_none=True)
            good=0

        if bi%500==0 or bi==len(train_loader):
            elapsed=(time.time()-t0)/60
            eta=elapsed*(len(train_loader)-bi)/max(1,bi)
            rs=rank_allocator.rank_summary()
            print(
                f"epoch={epoch} batch={bi}/{len(train_loader)} "
                f"bce={running/max(1,bi):.4f} "
                f"rank={rs['total']} "
                f"elapsed={elapsed:.1f}m ETA={eta:.1f}m",
                flush=True
            )

    # One DEV inference pass per epoch.
    with ema.applied():
        evaluate_epoch(epoch,"ema")

    gc.collect()
    if DEVICE.type=="cuda":
        torch.cuda.empty_cache()

assert BEST.exists()
print(
    "✅ AdaLoRA training complete | "
    "best clean-DEV CV Macro-F1:",best_score
)


AdaLoRA trainable: 860496
Head trainable   : 679653
batches/epoch: 10688
optimizer updates: 5344
AdaLoRA global rank budget: 336 -> 224
OOM fallback: MICRO_BATCH_SIZE=2, GRAD_ACCUM_STEPS=8.
epoch=1 batch=500/10688 bce=0.5339 rank=336 elapsed=0.9m ETA=17.6m
epoch=1 batch=1000/10688 bce=0.4817 rank=336 elapsed=1.8m ETA=17.0m
epoch=1 batch=1500/10688 bce=0.4560 rank=336 elapsed=2.6m ETA=16.1m
epoch=1 batch=2000/10688 bce=0.4345 rank=336 elapsed=3.5m ETA=15.3m
epoch=1 batch=2500/10688 bce=0.4196 rank=328 elapsed=4.4m ETA=14.4m
epoch=1 batch=3000/10688 bce=0.4099 rank=318 elapsed=5.3m ETA=13.6m
epoch=1 batch=3500/10688 bce=0.4013 rank=308 elapsed=6.2m ETA=12.7m
epoch=1 batch=4000/10688 bce=0.3939 rank=299 elapsed=7.1m ETA=11.8m
epoch=1 batch=4500/10688 bce=0.3894 rank=291 elapsed=8.0m ETA=10.9m
epoch=1 batch=5000/10688 bce=0.3836 rank=283 elapsed=8.8m ETA=10.1m
epoch=1 batch=5500/10688 bce=0.3790 rank=276 elapsed=9.7m ETA=9.2m
epoch=1 batch=6000/10688 bce=0.3757 rank=270 elapsed=10.6m ETA=8

In [16]:
# CELL 13 — reload BEST AdaLoRA state + final DEV diagnostics
best=torch.load(BEST,map_location="cpu",weights_only=False)
load_trainable_state(model,best["trainable_state"])
load_adalora_mask_state(model,best["adalora_masks"])
model.to(DEVICE)
model.eval()

thresholds=np.asarray(best["thresholds"],dtype=np.float32)

print("BEST epoch:",best["epoch"])
print("threshold strategy:",best["threshold_strategy"])
print("CV macro-F1:",best["cv_macro_f1"])
print("clean metrics:",best["clean_metrics"])
print("official metrics:",best["official_metrics"])
print("rank summary:",best["rank_summary"])
print(
    "threshold min/median/max:",
    thresholds.min(),np.median(thresholds),thresholds.max()
)

dids,dprob,dy,datt=predict_loader(model,official_loader)

cids=[dids[i] for i in CLEAN_IDX]
cprob=dprob[CLEAN_IDX]
re_pred=make_pred_records(
    cids,cprob,thresholds,True
)
re_score=exact_phase5_score(clean_truth,re_pred)
print("re-scored clean DEV:",re_score)

assert abs(
    re_score["macro_f1"]-best["clean_metrics"]["macro_f1"]
)<5e-3

print(
    "mean attention [HEAD,EVIDENCE,TAIL]:",
    datt[CLEAN_IDX].mean(0)
)
print(
    "mean predicted labels:",
    apply_thresholds(cprob,thresholds,True).sum(1).mean(),
    "gold:",dy[CLEAN_IDX].sum(1).mean()
)

np.savez_compressed(
    ROOT/"dev_probs_exp3.npz",
    id=np.asarray(dids),
    prob=dprob,
    y=dy,
    clean_idx=CLEAN_IDX,
    thresholds=thresholds,
)


BEST epoch: 2
threshold strategy: shrunk_tau_25
CV macro-F1: 0.29252955528952407
clean metrics: {'macro_f1': 0.3042064403376667, 'micro_f1': 0.40121077073247685, 'subset_accuracy': 0.016297326840902314}
official metrics: {'macro_f1': 0.3039324982144088, 'micro_f1': 0.401147859922179, 'subset_accuracy': 0.016501325999410666}
rank summary: {'min': 0, 'median': 8.0, 'max': 12, 'total': 224, 'per_module': [2, 8, 6, 7, 3, 8, 9, 8, 5, 5, 8, 10, 8, 5, 11, 11, 9, 11, 12, 12, 12, 11, 12, 11, 12, 2, 6, 0]}
threshold min/median/max: 0.24957296 0.4818204 0.74971265
re-scored clean DEV: {'macro_f1': 0.3042064403376667, 'micro_f1': 0.40121077073247685, 'subset_accuracy': 0.016297326840902314}
mean attention [HEAD,EVIDENCE,TAIL]: [0.3438349  0.27270254 0.38346156]
mean predicted labels: 6.167246348007552 gold: 3.9115571897048595


In [17]:
# CELL 14 — exact official Phase-5 evaluator + literal DEV script test
EVAL_SCRIPT = ROOT / "evaluate_phase5.py"
EVAL_SCRIPT.write_text('#!/usr/bin/env python3\n"""Local evaluator for Phase 5.\n\nTruth JSON: a list of objects, [{"id":"...", "gold":[0,2,5,...]}, ...]\nPrediction JSON: [{"id":"...", "pred":[0,2,5,...]}, ...]\n"""\nimport argparse,json,sys\nimport numpy as np\nfrom sklearn.metrics import accuracy_score, f1_score\n\ndef read_truth(path):\n    try:\n        x=json.load(open(path,encoding="utf-8"))\n    except json.JSONDecodeError as e:\n        raise ValueError(f"Truth must be a JSON file containing a list of records: {e}")\n    if not isinstance(x,list) or not x: raise ValueError(\'Truth must be a non-empty JSON list.\')\n    out={}; max_label=-1\n    for n,r in enumerate(x):\n        if not isinstance(r,dict) or \'id\' not in r or \'gold\' not in r: raise ValueError(f\'Truth item {n} must contain id and gold.\')\n        if not isinstance(r[\'id\'],str) or not r[\'id\'].strip() or not isinstance(r[\'gold\'],list): raise ValueError(f\'Invalid truth item {n}.\')\n        if r[\'id\'] in out: raise ValueError(f\'Duplicate truth ID: {r["id"]}\')\n        seen=set()\n        for label in r[\'gold\']:\n            if isinstance(label,bool) or not isinstance(label,int) or label<0 or label in seen: raise ValueError(f\'Invalid/duplicate label in truth item {n}.\')\n            seen.add(label); max_label=max(max_label,label)\n        out[r[\'id\']]=r[\'gold\']\n    return out,max_label+1\n\ndef read_pred(path,nlab):\n    x=json.load(open(path,encoding=\'utf-8\'))\n    if not isinstance(x,list) or not x: raise ValueError(\'Prediction must be a non-empty JSON list.\')\n    out={}\n    for n,r in enumerate(x):\n        if not isinstance(r,dict) or \'id\' not in r or \'pred\' not in r: raise ValueError(f\'Prediction item {n} must contain id and pred.\')\n        if not isinstance(r[\'id\'],str) or not isinstance(r[\'pred\'],list) or r[\'id\'] in out: raise ValueError(f\'Invalid or duplicate prediction item {n}.\')\n        seen=set()\n        for x in r[\'pred\']:\n            if isinstance(x,bool) or not isinstance(x,int) or x<0 or x>=nlab or x in seen: raise ValueError(f\'Invalid/duplicate label in prediction item {n}.\')\n            seen.add(x)\n        out[r[\'id\']]=r[\'pred\']\n    return out\n\ndef main():\n    ap=argparse.ArgumentParser(description=\'Evaluate Phase 5 multilabel predictions on your own held-out split.\')\n    ap.add_argument(\'--truth\',required=True); ap.add_argument(\'--prediction\',required=True); a=ap.parse_args()\n    t,n=read_truth(a.truth); p=read_pred(a.prediction,n)\n    if set(t)!=set(p): raise ValueError(f"ID mismatch: missing={len(set(t)-set(p))}, extra={len(set(p)-set(t))}")\n    yt=np.zeros((len(t),n),dtype=np.uint8); yp=np.zeros((len(t),n),dtype=np.uint8)\n    for i,k in enumerate(t):\n        for x in t[k]: yt[i,x]=1\n        for x in p[k]: yp[i,x]=1\n    print(f\'Examples: {len(t)}\'); print(f\'Labels: {n}\')\n    print(f\'Macro-F1: {f1_score(yt,yp,average="macro",zero_division=0):.6f}\')\n    micro=f1_score(yt,yp,average=\'micro\',zero_division=0)\n    subset=accuracy_score(yt,yp)\n    print(f\'Micro-F1: {micro:.6f}\')\n    print(f\'Subset Accuracy: {subset:.6f}\')\nif __name__==\'__main__\':\n    try: main()\n    except Exception as e: print(f\'EVALUATION FAILED: {e}\',file=sys.stderr); sys.exit(1)\n', encoding="utf-8")
print("✅ wrote exact evaluator:", EVAL_SCRIPT)


✅ wrote exact evaluator: /kaggle/working/phase5_exp3_model2_adalora_fast_v2_balanced/evaluate_phase5.py


In [18]:
# CELL 15 — HELDOUT TEST inference + base ZIP + exact parser validation
tids,tprob,_,tatt=predict_loader(model,test_loader)
pred_records=make_pred_records(
    tids,tprob,thresholds,True
)

assert len(pred_records)==len(test_records)
assert len({r["id"] for r in pred_records})==len(test_records)
assert {r["id"] for r in pred_records}=={
    str(r["id"]) for r in test_records
}

for r in pred_records:
    assert len(r["pred"])==len(set(r["pred"]))
    assert all(
        isinstance(x,int)
        and not isinstance(x,bool)
        and 0<=x<N_LABELS
        for x in r["pred"]
    )

PRED=ROOT/"prediction.json"
json.dump(
    pred_records,
    open(PRED,"w",encoding="utf-8"),
    ensure_ascii=False
)

ZIP=ROOT/"submission_exp3_Model2_AdaLoRA_HELDOUT.zip"
with zipfile.ZipFile(
    ZIP,"w",compression=zipfile.ZIP_DEFLATED
) as z:
    z.write(PRED,arcname="prediction.json")

print(
    "TEST mean predicted labels/document:",
    np.mean([len(r["pred"]) for r in pred_records])
)
print("TEST attention mean [HEAD,EVIDENCE,TAIL]:",tatt.mean(0))
print("ZIP:",ZIP)

np.savez_compressed(
    ROOT/"test_probs_heldout_exp3.npz",
    id=np.asarray(tids),
    prob=tprob,
    thresholds=thresholds
)

spec=importlib.util.spec_from_file_location(
    "evaluate_phase5_exact",EVAL_SCRIPT
)
eval_exact=importlib.util.module_from_spec(spec)
spec.loader.exec_module(eval_exact)

check_dir=ROOT/"zip_check_heldout"
if check_dir.exists():
    shutil.rmtree(check_dir)
check_dir.mkdir(parents=True,exist_ok=True)

with zipfile.ZipFile(ZIP,"r") as z:
    assert z.namelist()==["prediction.json"]
    z.extract("prediction.json",check_dir)

zip_pred=eval_exact.read_pred(
    str(check_dir/"prediction.json"),N_LABELS
)
expected_ids={str(r["id"]) for r in test_records}
assert set(zip_pred)==expected_ids
print("✅ HELDOUT ZIP PASS")


TEST mean predicted labels/document: 6.03187648820954
TEST attention mean [HEAD,EVIDENCE,TAIL]: [0.34450427 0.27082136 0.38467422]
ZIP: /kaggle/working/phase5_exp3_model2_adalora_fast_v2_balanced/submission_exp3_Model2_AdaLoRA_HELDOUT.zip
✅ HELDOUT ZIP PASS


## Fresh TRAIN+DEV AdaLoRA refit + heldout/refit ensemble

The refit starts again from the **original Phase-1 Model2 checkpoint**, injects fresh AdaLoRA triplets, and reruns the adaptive rank schedule on TRAIN+DEV.

It does not perform repeated DEV evaluation. Thresholds remain those selected by the honest heldout run.


In [19]:
# CELL 16 — fresh TRAIN+DEV AdaLoRA refit (no validation passes)
if not RUN_REFIT:
    print("RUN_REFIT=False -> fresh TRAIN+DEV refit skipped.")
else:
    REFIT_EPOCHS=max(1,int(best["epoch"]))
    REFIT_SEED=SEED+777

    print("Fresh refit epochs:",REFIT_EPOCHS)
    print("TRAIN+DEV rows:",len(train_records)+len(dev_raw))
    print(
        "Heldout/refit ensemble weights:",
        1.0-REFIT_ENSEMBLE_WEIGHT,"/",REFIT_ENSEMBLE_WEIGHT
    )

    torch.manual_seed(REFIT_SEED)
    np.random.seed(REFIT_SEED)
    random.seed(REFIT_SEED)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(REFIT_SEED)

    refit_backbone=build_backbone()
    obj_refit=torch.load(
        MODEL2_CKPT,map_location="cpu",weights_only=False
    )
    missing,unexpected=refit_backbone.load_state_dict(
        obj_refit["model"],strict=False
    )
    assert not missing and not unexpected,(missing,unexpected)

    for p in refit_backbone.parameters():
        p.requires_grad=False

    refit_injected=inject_adalora(refit_backbone)
    assert len(refit_injected)==len(injected_modules)

    refit_model=StatuteAdaLoRAClassifier(
        refit_backbone
    ).to(DEVICE)

    refit_adalora_params=[]
    refit_head_params=[]
    for n,p in refit_model.named_parameters():
        if not p.requires_grad:
            continue
        if n.startswith("backbone.") and any(
            k in n for k in [".P",".E",".Q"]
        ):
            refit_adalora_params.append(p)
        else:
            refit_head_params.append(p)

    assert refit_adalora_params and refit_head_params
    print(
        "refit AdaLoRA trainable:",
        sum(p.numel() for p in refit_adalora_params)
    )
    print(
        "refit head trainable:",
        sum(p.numel() for p in refit_head_params)
    )

    refit_train_ds=torch.utils.data.ConcatDataset([
        LSIDataset(
            train_records,train_tok,train_lens,labeled=True
        ),
        LSIDataset(
            dev_raw,official_tok,official_lens,labeled=True
        ),
    ])

    refit_gen=torch.Generator()
    refit_gen.manual_seed(REFIT_SEED)

    refit_loader=DataLoader(
        refit_train_ds,
        batch_size=MICRO_BATCH_SIZE,
        shuffle=True,
        num_workers=NUM_WORKERS,
        pin_memory=True,
        persistent_workers=(NUM_WORKERS>0),
        generator=refit_gen,
    )

    full_counts=np.zeros(N_LABELS,dtype=np.int64)
    for r in train_records:
        full_counts[r["labels"]]+=1
    for r in dev_raw:
        full_counts[r["labels"]]+=1

    full_n=len(train_records)+len(dev_raw)
    full_raw_pw=(full_n-full_counts)/full_counts
    REFIT_POS_WEIGHT=np.clip(
        np.sqrt(full_raw_pw),1.0,10.0
    ).astype(np.float32)
    refit_pos_weight_t=torch.tensor(
        REFIT_POS_WEIGHT,dtype=torch.float32,device=DEVICE
    )

    refit_optimizer=torch.optim.AdamW(
        [
            {
                "params":refit_adalora_params,
                "lr":ADALORA_LR,
                "weight_decay":0.0
            },
            {
                "params":refit_head_params,
                "lr":HEAD_LR,
                "weight_decay":WEIGHT_DECAY
            },
        ],
        betas=(0.9,0.999),eps=1e-8
    )

    refit_updates_per_epoch=math.ceil(
        len(refit_loader)/GRAD_ACCUM_STEPS
    )
    refit_total_updates=(
        refit_updates_per_epoch*REFIT_EPOCHS
    )
    refit_warmup=max(
        1,int(WARMUP_RATIO*refit_total_updates)
    )

    def refit_lr_lambda(step):
        if step<refit_warmup:
            return (step+1)/refit_warmup
        q=(step-refit_warmup)/max(
            1,refit_total_updates-refit_warmup
        )
        return max(
            0.08,0.5*(1+math.cos(math.pi*min(1.0,q)))
        )

    refit_scheduler=torch.optim.lr_scheduler.LambdaLR(
        refit_optimizer,refit_lr_lambda
    )
    refit_rank_allocator=AdaRankAllocator(
        model=refit_model,
        total_steps=refit_total_updates,
        init_rank=ADALORA_INIT_RANK,
        target_avg_rank=ADALORA_TARGET_AVG_RANK,
        warmup_frac=ADALORA_WARMUP_FRAC,
        final_frac=ADALORA_FINAL_FRAC,
        interval=ADALORA_MASK_INTERVAL,
    )
    refit_scaler=torch.amp.GradScaler(
        "cuda",enabled=(DEVICE.type=="cuda" and USE_AMP)
    )
    refit_ema=EMA(refit_model,EMA_DECAY)

    REFIT_STATE=ROOT/"refit_exp3_adalora_state.pt"

    refit_optimizer.zero_grad(set_to_none=True)
    refit_global_update=0

    for epoch in range(1,REFIT_EPOCHS+1):
        refit_model.train()
        t0=time.time()
        running=0.0
        good=0

        for bi,b in enumerate(refit_loader,1):
            x=b["input_ids"].to(DEVICE,non_blocking=True)
            v=b["valid"].to(DEVICE,non_blocking=True)
            y=b["labels"].to(DEVICE,non_blocking=True)

            with torch.amp.autocast(
                "cuda",dtype=torch.float16,
                enabled=(DEVICE.type=="cuda" and USE_AMP)
            ):
                logits,_=refit_model(x,v)
                bce=F.binary_cross_entropy_with_logits(
                    logits,y,pos_weight=refit_pos_weight_t
                )
                orth=adalora_orth_loss(refit_model)
                loss=bce+ADALORA_ORTH_COEF*orth
                scaled_loss=loss/GRAD_ACCUM_STEPS

            if torch.isfinite(loss):
                refit_scaler.scale(scaled_loss).backward()
                running+=float(bce.detach())
                good+=1
            else:
                print("⚠️ refit non-finite loss",bi)

            if (
                bi%GRAD_ACCUM_STEPS==0
                or bi==len(refit_loader)
            ):
                if good:
                    refit_scaler.unscale_(refit_optimizer)

                    refit_rank_allocator.update_importance()

                    torch.nn.utils.clip_grad_norm_(
                        [
                            p for p in refit_model.parameters()
                            if p.requires_grad
                        ],
                        GRAD_CLIP
                    )

                    old_scale=float(refit_scaler.get_scale())
                    refit_scaler.step(refit_optimizer)
                    refit_scaler.update()
                    refit_optimizer.zero_grad(set_to_none=True)

                    if (
                        float(refit_scaler.get_scale())
                        >=old_scale
                    ):
                        refit_scheduler.step()
                        refit_global_update+=1
                        refit_ema.update()
                        refit_rank_allocator.reallocate(
                            refit_global_update
                        )
                else:
                    refit_optimizer.zero_grad(set_to_none=True)
                good=0

            if bi%500==0 or bi==len(refit_loader):
                elapsed=(time.time()-t0)/60
                eta=elapsed*(
                    len(refit_loader)-bi
                )/max(1,bi)
                rs=refit_rank_allocator.rank_summary()
                print(
                    f"REFIT epoch={epoch}/{REFIT_EPOCHS} "
                    f"batch={bi}/{len(refit_loader)} "
                    f"bce={running/max(1,bi):.4f} "
                    f"rank={rs['total']} "
                    f"elapsed={elapsed:.1f}m ETA={eta:.1f}m",
                    flush=True
                )

    with refit_ema.applied():
        atomic_torch_save(
            {
                "trainable_state":get_trainable_state(
                    refit_model
                ),
                "adalora_masks":get_adalora_mask_state(
                    refit_model
                ),
                "rank_summary":
                    refit_rank_allocator.rank_summary(),
                "epochs":REFIT_EPOCHS,
                "global_updates":refit_global_update,
                "model2_sha256":sha256_file(MODEL2_CKPT),
                "tokenizer_sha256":
                    sha256_file(TOKENIZER_FILE),
                "thresholds_from_heldout":thresholds,
            },
            REFIT_STATE
        )

        rids,rprob,_,ratt=predict_loader(
            refit_model,test_loader
        )

    assert list(map(str,rids))==list(map(str,tids))

    np.savez_compressed(
        ROOT/"test_probs_refit_exp3.npz",
        id=np.asarray(rids),
        prob=rprob,
        thresholds=thresholds
    )

    print("✅ fresh TRAIN+DEV AdaLoRA refit complete")
    print(
        "refit final rank summary:",
        refit_rank_allocator.rank_summary()
    )


Fresh refit epochs: 2
TRAIN+DEV rows: 52931
Heldout/refit ensemble weights: 0.5 / 0.5
refit AdaLoRA trainable: 860496
refit head trainable: 679653
REFIT epoch=1/2 batch=500/13233 bce=0.5652 rank=336 elapsed=0.9m ETA=22.6m
REFIT epoch=1/2 batch=1000/13233 bce=0.5049 rank=336 elapsed=1.8m ETA=21.7m
REFIT epoch=1/2 batch=1500/13233 bce=0.4725 rank=336 elapsed=2.7m ETA=20.8m
REFIT epoch=1/2 batch=2000/13233 bce=0.4491 rank=336 elapsed=3.5m ETA=19.9m
REFIT epoch=1/2 batch=2500/13233 bce=0.4329 rank=336 elapsed=4.4m ETA=19.0m
REFIT epoch=1/2 batch=3000/13233 bce=0.4204 rank=330 elapsed=5.3m ETA=18.1m
REFIT epoch=1/2 batch=3500/13233 bce=0.4108 rank=322 elapsed=6.2m ETA=17.3m
REFIT epoch=1/2 batch=4000/13233 bce=0.4041 rank=313 elapsed=7.1m ETA=16.4m
REFIT epoch=1/2 batch=4500/13233 bce=0.3977 rank=306 elapsed=8.0m ETA=15.5m
REFIT epoch=1/2 batch=5000/13233 bce=0.3921 rank=298 elapsed=8.9m ETA=14.6m
REFIT epoch=1/2 batch=5500/13233 bce=0.3874 rank=292 elapsed=9.8m ETA=13.7m
REFIT epoch=1/2 ba

In [20]:
# CELL 17 — REFIT + HELDOUT/REFIT ENSEMBLE ZIPs
def validate_submission_zip(zip_path,expected_ids):
    tmp=ROOT/("check_"+zip_path.stem)
    if tmp.exists():
        shutil.rmtree(tmp)
    tmp.mkdir(parents=True,exist_ok=True)

    with zipfile.ZipFile(zip_path,"r") as z:
        names=z.namelist()
        assert names==["prediction.json"],names
        z.extract("prediction.json",tmp)

    parsed=eval_exact.read_pred(
        str(tmp/"prediction.json"),N_LABELS
    )
    assert set(parsed)==expected_ids
    return parsed

expected_ids={str(r["id"]) for r in test_records}

if not RUN_REFIT:
    print("RUN_REFIT=False -> refit/ensemble ZIPs skipped. Base heldout ZIP:", ZIP)
else:
    # Refit-only.
    refit_records=make_pred_records(
        rids,rprob,thresholds,True
    )
    REFIT_PRED=ROOT/"prediction_refit.json"
    json.dump(
        refit_records,
        open(REFIT_PRED,"w",encoding="utf-8"),
        ensure_ascii=False
    )
    REFIT_ZIP=ROOT/"submission_exp3_Model2_AdaLoRA_REFIT.zip"
    with zipfile.ZipFile(
        REFIT_ZIP,"w",compression=zipfile.ZIP_DEFLATED
    ) as z:
        z.write(REFIT_PRED,arcname="prediction.json")

    validate_submission_zip(REFIT_ZIP,expected_ids)
    print("✅ REFIT ZIP PASS:",REFIT_ZIP)

    # Heldout + refit probability ensemble.
    ensemble_prob=(
        (1.0-REFIT_ENSEMBLE_WEIGHT)*tprob
        +REFIT_ENSEMBLE_WEIGHT*rprob
    )

    ensemble_records=make_pred_records(
        tids,ensemble_prob,thresholds,True
    )
    ENSEMBLE_PRED=ROOT/"prediction_heldout_refit_ensemble.json"
    json.dump(
        ensemble_records,
        open(ENSEMBLE_PRED,"w",encoding="utf-8"),
        ensure_ascii=False
    )

    ENSEMBLE_ZIP=ROOT/"submission_exp3_Model2_AdaLoRA_HELDOUT_REFIT_ENSEMBLE.zip"
    with zipfile.ZipFile(
        ENSEMBLE_ZIP,"w",
        compression=zipfile.ZIP_DEFLATED
    ) as z:
        z.write(
            ENSEMBLE_PRED,arcname="prediction.json"
        )

    validate_submission_zip(
        ENSEMBLE_ZIP,expected_ids
    )

    np.savez_compressed(
        ROOT/"test_probs_heldout_refit_ensemble_exp3.npz",
        id=np.asarray(tids),
        prob=ensemble_prob,
        thresholds=thresholds,
        refit_weight=REFIT_ENSEMBLE_WEIGHT
    )

    print("✅ HELDOUT+REFIT ENSEMBLE ZIP PASS:",ENSEMBLE_ZIP)
    print("\nCORE SUBMISSION CANDIDATES")
    print("1. HELDOUT :",ZIP)
    print("2. REFIT   :",REFIT_ZIP)
    print("3. ENSEMBLE:",ENSEMBLE_ZIP)


✅ REFIT ZIP PASS: /kaggle/working/phase5_exp3_model2_adalora_fast_v2_balanced/submission_exp3_Model2_AdaLoRA_REFIT.zip
✅ HELDOUT+REFIT ENSEMBLE ZIP PASS: /kaggle/working/phase5_exp3_model2_adalora_fast_v2_balanced/submission_exp3_Model2_AdaLoRA_HELDOUT_REFIT_ENSEMBLE.zip

CORE SUBMISSION CANDIDATES
1. HELDOUT : /kaggle/working/phase5_exp3_model2_adalora_fast_v2_balanced/submission_exp3_Model2_AdaLoRA_HELDOUT.zip
2. REFIT   : /kaggle/working/phase5_exp3_model2_adalora_fast_v2_balanced/submission_exp3_Model2_AdaLoRA_REFIT.zip
3. ENSEMBLE: /kaggle/working/phase5_exp3_model2_adalora_fast_v2_balanced/submission_exp3_Model2_AdaLoRA_HELDOUT_REFIT_ENSEMBLE.zip


In [21]:
# CELL 18 — manifest for report/viva
manifest={
    "phase":5,
    "experiment":3,
    "version":NOTEBOOK_VERSION,
    "task":"Legal Statute Identification",
    "method":"AdaLoRA — Adaptive Budget Allocation for PEFT",
    "phase1_model":"Model2 decoder-only causal Transformer",
    "architecture":{
        "vocab_size":VOCAB_SIZE,
        "d_model":D_MODEL,
        "n_layers":N_LAYERS,
        "n_heads":N_HEADS,
        "ffn_dim":FFN_DIM,
        "max_positions":MAX_POSITIONS,
        "sinusoidal_positions":True,
        "pre_layernorm":True,
        "activation":"GELU",
        "tied_output_embedding":True,
    },
    "adalora":{
        "parameterization":"P diag(E) Q",
        "init_rank":ADALORA_INIT_RANK,
        "target_avg_rank":ADALORA_TARGET_AVG_RANK,
        "alpha":ADALORA_ALPHA,
        "adapt_start_block":ADAPT_START_BLOCK,
        "adapted_blocks":ADAPTED_BLOCKS,
        "targets":list(ADALORA_TARGETS),
        "budget_schedule":"warmup + cubic decay + final fixed allocation",
        "warmup_frac":ADALORA_WARMUP_FRAC,
        "final_frac":ADALORA_FINAL_FRAC,
        "mask_interval":ADALORA_MASK_INTERVAL,
        "best_rank_summary":best["rank_summary"],
        "backbone_trainable_params":backbone_trainable,
    },
    "speed_design":{
        "n_chunks":N_CHUNKS,
        "chunk_len":CHUNK_LEN,
        "views":VIEW_NAMES,
        "evals_per_epoch":1,
        "epochs":EPOCHS,
        "frozen_prefix_blocks":ADAPT_START_BLOCK,
        "refit":RUN_REFIT,
        "refit_epochs":globals().get("REFIT_EPOCHS"),
        "heldout_refit_ensemble_weight":
            REFIT_ENSEMBLE_WEIGHT,
    },
    "training":{
        "loss":"BCEWithLogits + sqrt(raw pos_weight) clipped [1,10] + AdaLoRA orthogonality",
        "adalora_lr":ADALORA_LR,
        "head_lr":HEAD_LR,
        "micro_batch":MICRO_BATCH_SIZE,
        "grad_accum":GRAD_ACCUM_STEPS,
        "ema_decay":EMA_DECAY,
    },
    "selection":{
        "primary":"cross-fitted clean-DEV Macro-F1",
        "best_epoch":best["epoch"],
        "cv_macro_f1":best["cv_macro_f1"],
        "threshold_strategy":best[
            "threshold_strategy"
        ],
        "clean_metrics":best["clean_metrics"],
        "official_metrics":best["official_metrics"],
    },
    "submission_variants":{
        "heldout_zip":str(ZIP),
        "refit_zip":str(globals().get("REFIT_ZIP")),
        "heldout_refit_ensemble_zip":
            str(globals().get("ENSEMBLE_ZIP")),
    },
    "assets":{
        "model2_sha256":
            sha256_file(MODEL2_CKPT),
        "tokenizer_sha256":
            sha256_file(TOKENIZER_FILE),
        "dataset_sha256":DATA_SHA256,
    },
    "reference":{
        "paper":"Zhang et al. (2023), Adaptive Budget Allocation for Parameter-Efficient Fine-Tuning",
        "venue":"ICLR 2023",
        "arxiv":"2303.10512",
    },
}

json.dump(
    manifest,
    open(ROOT/"manifest.json","w"),
    indent=2
)
print(json.dumps(manifest,indent=2))

print("\nArtifacts:")
for p in sorted(ROOT.iterdir()):
    if p.is_file():
        print(
            " ",p.name,
            round(p.stat().st_size/1e6,2),"MB"
        )


{
  "phase": 5,
  "experiment": 3,
  "version": "exp3_model2_adalora_fast_v2_balanced_refit_ensemble",
  "task": "Legal Statute Identification",
  "method": "AdaLoRA \u2014 Adaptive Budget Allocation for PEFT",
  "phase1_model": "Model2 decoder-only causal Transformer",
  "architecture": {
    "vocab_size": 24576,
    "d_model": 640,
    "n_layers": 13,
    "n_heads": 10,
    "ffn_dim": 2560,
    "max_positions": 2048,
    "sinusoidal_positions": true,
    "pre_layernorm": true,
    "activation": "GELU",
    "tied_output_embedding": true
  },
  "adalora": {
    "parameterization": "P diag(E) Q",
    "init_rank": 12,
    "target_avg_rank": 8,
    "alpha": 16.0,
    "adapt_start_block": 6,
    "adapted_blocks": [
      6,
      7,
      8,
      9,
      10,
      11,
      12
    ],
    "targets": [
      "qkv",
      "out_proj",
      "ff_in",
      "ff_out"
    ],
    "budget_schedule": "warmup + cubic decay + final fixed allocation",
    "warmup_frac": 0.1,
    "final_frac": 0.2,
   

## What to record after the run

Keep these values for your report/viva:

- Model2 total parameter count
- AdaLoRA trainable parameter count
- initial and final global rank budgets
- final rank distribution across adapted matrices
- epoch wall-clock time
- clean and official DEV Macro-F1
- Micro-F1 and subset accuracy
- threshold strategy
- heldout / refit / ensemble Codabench scores
- comparison against Model4 Partial FT and Model4 LoRMA+

The defining difference is **adaptive SVD-like rank-budget allocation**, not merely another fixed-rank LoRA configuration.


In [22]:
# CELL 19 (OPTIONAL) — blend Exp2 + Exp3 probabilities without retraining
# Upload Exp2:
#   dev_probs_exp2.npz
#   test_probs_heldout_refit_ensemble.npz  (or a test-prob file of your choice)
# If files are not found, this cell safely skips.

EXP2_DEV_NPZ=None
EXP2_TEST_NPZ=None

def _find_named_npz(names):
    hits=[]
    for base in ["/kaggle/input","/kaggle/working"]:
        if not os.path.exists(base):
            continue
        for root,_,files in os.walk(base):
            for name in names:
                if name in files:
                    p=os.path.join(root,name)
                    if "phase5_exp3_model2_adalora" not in p.lower():
                        hits.append(p)
    return sorted(set(hits))[0] if hits else None

EXP2_DEV_NPZ=EXP2_DEV_NPZ or _find_named_npz([
    "dev_probs_exp2.npz"
])
EXP2_TEST_NPZ=EXP2_TEST_NPZ or _find_named_npz([
    "test_probs_heldout_refit_ensemble.npz",
    "test_probs_exp2.npz"
])

print("Exp2 DEV :",EXP2_DEV_NPZ)
print("Exp2 TEST:",EXP2_TEST_NPZ)

if not (EXP2_DEV_NPZ and EXP2_TEST_NPZ):
    print("⚠️ Exp2 probability files not found -> optional blend skipped.")
else:
    d2=np.load(EXP2_DEV_NPZ,allow_pickle=True)
    t2=np.load(EXP2_TEST_NPZ,allow_pickle=True)

    def _align(ids_src,prob_src,ids_dst,label):
        pos={str(i):k for k,i in enumerate(ids_src)}
        missing=[
            str(i) for i in ids_dst
            if str(i) not in pos
        ]
        assert not missing,(
            label,len(missing),missing[:5]
        )
        return prob_src[
            [pos[str(i)] for i in ids_dst]
        ]

    p2_dev=_align(
        d2["id"],d2["prob"],dids,"Exp2 DEV"
    )
    p2_test=_align(
        t2["id"],t2["prob"],tids,"Exp2 TEST"
    )

    cy=dy[CLEAN_IDX]
    rows=[]
    for w in np.round(
        np.arange(0.0,1.01,0.1),2
    ): # w = Exp3 heldout weight
        pb=w*dprob+(1.0-w)*p2_dev
        tune_w,_,_=optimize_thresholds(
            cy,pb[CLEAN_IDX]
        )
        rows.append(
            (
                float(w),
                tune_w["cv_macro_f1"],
                tune_w["name"]
            )
        )
        print(
            f"w_exp3={w:.1f} "
            f"CV={tune_w['cv_macro_f1']:.4f} "
            f"{tune_w['name']}"
        )

    w_best,best_cv,_=max(
        rows,key=lambda r:r[1]
    )
    print(
        "✅ best Exp3 blend weight:",
        w_best,"CV:",best_cv
    )

    pb_dev=w_best*dprob+(1.0-w_best)*p2_dev
    tune_b,_,_=optimize_thresholds(
        cy,pb_dev[CLEAN_IDX]
    )
    thr_b=np.asarray(
        tune_b["thresholds"],dtype=np.float32
    )

    exp3_test_prob = ensemble_prob if RUN_REFIT else tprob
    pb_test=(
        w_best*exp3_test_prob
        +(1.0-w_best)*p2_test
    )
    blend_records=make_pred_records(
        tids,pb_test,thr_b,True
    )

    BLEND_PRED=ROOT/"prediction_exp2_exp3_blend.json"
    json.dump(
        blend_records,
        open(BLEND_PRED,"w",encoding="utf-8"),
        ensure_ascii=False
    )

    BLEND_ZIP=ROOT/"submission_exp2_exp3_BLEND.zip"
    with zipfile.ZipFile(
        BLEND_ZIP,"w",
        compression=zipfile.ZIP_DEFLATED
    ) as z:
        z.write(
            BLEND_PRED,
            arcname="prediction.json"
        )

    validate_submission_zip(
        BLEND_ZIP,expected_ids
    )
    print("✅ EXP2+EXP3 BLEND ZIP PASS:",BLEND_ZIP)


Exp2 DEV : None
Exp2 TEST: None
⚠️ Exp2 probability files not found -> optional blend skipped.
